# 11_deployment_prep.ipynb

## Project: Fabric Defect Detection
### Part 11: Deployment Preparation (No Scikit-Learn)

**Goal:** Package the custom models into a single portable file and generate a standalone inference script.

In [1]:
import os
import sys
import numpy as np
import pickle
import glob
import torch
import torchvision.models as models
import torch.nn as nn
from PIL import Image
from datetime import datetime

# -----------------------------------------------------------
# 1. SETUP & PATHS
# -----------------------------------------------------------
try:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive', force_remount=True)
    BASE_DIR = "/content/drive/MyDrive/FabricDefect"
except ImportError:
    BASE_DIR = "./FabricDefect"

MODEL_DIR = f"{BASE_DIR}/models"
BACKUP_DIR = f"{BASE_DIR}/backups"
DEPLOY_DIR = f"{BASE_DIR}/deployment"
FEATURE_DIR = f"{BASE_DIR}/features"

os.makedirs(DEPLOY_DIR, exist_ok=True)
print(f"✓ Setup Complete. Deployment artifacts will be saved to: {DEPLOY_DIR}")

Mounted at /content/drive
✓ Setup Complete. Deployment artifacts will be saved to: /content/drive/MyDrive/FabricDefect/deployment


## 2. REDEFINE MODEL CLASSES
Essential for `pickle` to reconstruct the objects correctly.

In [2]:
# -----------------------------------------------------------
# 2. REDEFINE MODEL CLASSES (MANDATORY FOR PICKLE)
# -----------------------------------------------------------
# These definitions must exist in the namespace for pickle to load the objects.
def softmax(z):
    z -= np.max(z, axis=1, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=1, keepdims=True)

class TreeNode:
    def __init__(self, depth=0):
        self.depth = depth; self.left = None; self.right = None; self.feature_idx = None; self.threshold = None; self.value = None; self.is_leaf = False

class RegularizedTree:
    def __init__(self, max_depth=3, lambda_reg=1.0, gamma=0.0, min_child_weight=1.0, feature_subsample=1.0):
        self.max_depth = max_depth; self.lambda_reg = lambda_reg; self.gamma = gamma; self.root = None
    def predict(self, X): return np.array([self._traverse(x, self.root) for x in X])
    def _traverse(self, x, node):
        if node.is_leaf: return node.value
        if x[node.feature_idx] <= node.threshold: return self._traverse(x, node.left)
        else: return self._traverse(x, node.right)

class CustomModelBase:
    def predict_proba(self, X):
        if not hasattr(self, 'n_classes'): self.n_classes = len(self.trees[0])
        scores = np.zeros((X.shape[0], self.n_classes))
        for iter_trees in self.trees:
            for k, tree in enumerate(iter_trees):
                scores[:, k] += self.lr * tree.predict(X)
        return softmax(scores)
    def predict(self, X): return np.argmax(self.predict_proba(X), axis=1)

class CustomXGBoost(CustomModelBase):
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=3, reg_lambda=1.0, gamma=0.0):
        self.K = n_estimators; self.lr = learning_rate; self.trees = []

class CustomLightGBM(CustomModelBase):
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=3, goss_alpha=0.2, goss_beta=0.2):
        self.K = n_estimators; self.lr = learning_rate; self.trees = []

class CustomCatBoost(CustomModelBase):
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=3):
        self.K = n_estimators; self.lr = learning_rate; self.trees = []

class WeightStacker:
    def __init__(self, n_trials=2000):
        self.n_trials = n_trials; self.best_weights = None

print("✓ Model Classes Defined.")

✓ Model Classes Defined.


In [3]:
# -----------------------------------------------------------
# 3. LOAD & PACKAGE MODELS (INTELLIGENT FALLBACKS)
# -----------------------------------------------------------
print("\nPackaging Model...")

def get_latest_folder(pattern):
    folders = glob.glob(os.path.join(BACKUP_DIR, pattern))
    return max(folders, key=os.path.getmtime) if folders else None

# Find the latest optimized run
opt_folder = get_latest_folder("optimized_run_*")

if not opt_folder:
    print("❌ No optimized models found! Please ensure Notebook 10 was run.")
else:
    print(f"   Loading from: {opt_folder}")

    models_dict = {}
    try:
        # Load the 4 model components
        for name in ['final_xgb', 'final_lgb', 'final_cat', 'final_stacker']:
            with open(f"{opt_folder}/{name}.pkl", "rb") as f:
                models_dict[name] = pickle.load(f)
        print("   ✓ Models loaded from backup.")
    except AttributeError as e:
        print(f"   ❌ Pickle Load Error: {e}")
        print("   (Ensure the class definitions above match exactly what was used in Notebook 10)")
        raise e

    # --- ROBUST LABEL LOADING STRATEGY ---
    idx_to_class = {}

    # Attempt 1: Load from dataset_metadata.pkl (Most reliable based on your files)
    try:
        with open(f"{FEATURE_DIR}/dataset_metadata.pkl", "rb") as f:
            meta = pickle.load(f)
        class_to_idx = meta.get('class_to_idx', {})
        idx_to_class = {v: k for k, v in class_to_idx.items()}
        print("   ✓ Loaded classes from dataset_metadata.pkl")
    except Exception:
        # Attempt 2: Load from label_encoder.pkl (if it existed)
        try:
            with open(f"{FEATURE_DIR}/label_encoder.pkl", "rb") as f:
                le = pickle.load(f)
            if hasattr(le, 'classes_'):
                 idx_to_class = {i: c for i, c in enumerate(le.classes_)}
            print("   ✓ Loaded classes from label_encoder.pkl")
        except Exception:
             print(f"   ⚠️ WARNING: Could not find class labels. Using numeric labels.")
             # Fallback to generic labels if all else fails
             idx_to_class = {0: "Defect 0", 1: "Defect 1", 2: "Defect 2", 3: "Defect 3", 4: "Defect 4", 5: "Defect 5"}

    print(f"   Classes: {list(idx_to_class.values())}")

    # Create and Save the Deployment Package
    deployment_package = {
        "models": models_dict,
        "idx_to_class": idx_to_class,
        "meta": {"created_at": datetime.now().isoformat()}
    }

    pkg_path = f"{DEPLOY_DIR}/deployment_model.pkl"
    with open(pkg_path, "wb") as f:
        pickle.dump(deployment_package, f)

    print(f"✅ Deployment Package Saved: {pkg_path}")


Packaging Model...
   Loading from: /content/drive/MyDrive/FabricDefect/backups/optimized_run_20251214_165423
   ✓ Models loaded from backup.
   ✓ Loaded classes from dataset_metadata.pkl
   Classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
✅ Deployment Package Saved: /content/drive/MyDrive/FabricDefect/deployment/deployment_model.pkl


In [4]:
# -----------------------------------------------------------
# 4. GENERATE INFERENCE SCRIPT (inference.py)
# Includes CustomUnpickler to fix "Can't get attribute" errors
# -----------------------------------------------------------
inference_script_content = """
import pickle
import numpy as np
import torch
import torchvision.models as models
import torch.nn as nn
from PIL import Image
import sys

# --- 1. MODEL CLASSES (Must match training exactly) ---
def softmax(z):
    z -= np.max(z, axis=1, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=1, keepdims=True)

class TreeNode:
    def __init__(self, depth=0):
        self.depth = depth; self.left = None; self.right = None; self.feature_idx = None; self.threshold = None; self.value = None; self.is_leaf = False

class RegularizedTree:
    def __init__(self, max_depth=3, lambda_reg=1.0, gamma=0.0, min_child_weight=1.0, feature_subsample=1.0):
        self.max_depth = max_depth; self.lambda_reg = lambda_reg; self.gamma = gamma; self.root = None
    def predict(self, X): return np.array([self._traverse(x, self.root) for x in X])
    def _traverse(self, x, node):
        if node.is_leaf: return node.value
        if x[node.feature_idx] <= node.threshold: return self._traverse(x, node.left)
        else: return self._traverse(x, node.right)

class CustomModelBase:
    def predict_proba(self, X):
        if not hasattr(self, 'n_classes'): self.n_classes = len(self.trees[0])
        scores = np.zeros((X.shape[0], self.n_classes))
        for iter_trees in self.trees:
            for k, tree in enumerate(iter_trees):
                scores[:, k] += self.lr * tree.predict(X)
        return softmax(scores)

class CustomXGBoost(CustomModelBase):
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=3, reg_lambda=1.0, gamma=0.0):
        self.K = n_estimators; self.lr = learning_rate; self.trees = []

class CustomLightGBM(CustomModelBase):
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=3, goss_alpha=0.2, goss_beta=0.2):
        self.K = n_estimators; self.lr = learning_rate; self.trees = []

class CustomCatBoost(CustomModelBase):
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=3):
        self.K = n_estimators; self.lr = learning_rate; self.trees = []

class WeightStacker:
    def __init__(self, n_trials=2000):
        self.n_trials = n_trials; self.best_weights = None

# --- 2. CUSTOM UNPICKLER (THE FIX) ---
class ModelUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        # Redirect __main__ class lookups to the current script's global namespace
        if module == "__main__":
            if name in globals():
                return globals()[name]
        return super().find_class(module, name)

# --- 3. PREDICTOR CLASS ---
class FabricPredictor:
    def __init__(self, model_path='deployment_model.pkl'):
        print(f"Loading model from {model_path}...")

        # Use Custom Unpickler to load safely
        with open(model_path, 'rb') as f:
            data = ModelUnpickler(f).load()

        self.models = data['models']
        self.idx_to_class = data['idx_to_class']

        # Load ResNet50
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.resnet = models.resnet50(weights="IMAGENET1K_V1")
        self.resnet.fc = nn.Identity()
        self.resnet.to(self.device)
        self.resnet.eval()

    def predict(self, img_pil):
        # 1. Preprocess
        img = img_pil.resize((224, 224)).convert('RGB')
        img_np = np.array(img, dtype=np.float32) / 255.0

        # Standard ImageNet normalization
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        img_np = (img_np - mean) / std

        img_tensor = torch.tensor(img_np.transpose(2, 0, 1)).unsqueeze(0).to(self.device)

        # 2. Extract Features
        with torch.no_grad():
            features = self.resnet(img_tensor).cpu().numpy()

        # 3. Base Predictions
        p1 = self.models['final_xgb'].predict_proba(features)
        p2 = self.models['final_lgb'].predict_proba(features)
        p3 = self.models['final_cat'].predict_proba(features)

        # 4. Ensemble Weighting
        w = self.models['final_stacker'].best_weights
        final_prob = (p1 * w[0]) + (p2 * w[1]) + (p3 * w[2])

        # 5. Result
        idx = np.argmax(final_prob)
        # Handle dict keys being strings or ints
        try:
            label = self.idx_to_class[idx]
        except KeyError:
            label = self.idx_to_class[str(idx)]

        conf = final_prob[0][idx]

        return label, conf, final_prob[0]
"""

with open(f"{DEPLOY_DIR}/inference.py", "w") as f:
    f.write(inference_script_content)
print("✅ Generated inference.py ")

✅ Generated inference.py 


In [5]:
# -----------------------------------------------------------
# 5. GENERATE STREAMLIT APP (app.py)
# -----------------------------------------------------------
app_code = """
import streamlit as st
from PIL import Image
import numpy as np
import pandas as pd
# Import the predictor class from our generated script
from inference import FabricPredictor

st.set_page_config(page_title="Fabric Defect Detector", page_icon="🧵")

st.title("🧵 Fabric Defect Detection System")
st.markdown("**Custom Ensemble Model** | *No Scikit-Learn Dependencies*")

@st.cache_resource
def load_model():
    return FabricPredictor('deployment_model.pkl')

try:
    model = load_model()
    st.success("Model Loaded Successfully!")
except Exception as e:
    st.error(f"Error loading model: {e}")
    st.stop()

uploaded_file = st.file_uploader("Choose an image...", type=['jpg', 'jpeg', 'png', 'bmp'])

if uploaded_file is not None:
    col1, col2 = st.columns(2)
    image = Image.open(uploaded_file)
    with col1:
        st.image(image, caption='Uploaded Fabric', use_column_width=True)

    with st.spinner('Analyzing...'):
        label, confidence, all_probs = model.predict(image)

    with col2:
        st.subheader(f"Result: {label}")
        st.metric(label="Confidence", value=f"{confidence:.2%}")

        # Prepare chart data
        # Ensure we display probabilities in the correct order of classes
        sorted_indices = np.argsort(list(model.idx_to_class.keys()))
        class_names = [model.idx_to_class[i] for i in sorted_indices]

        probs_df = pd.DataFrame({
            'Class': class_names,
            'Probability': all_probs
        })
        st.bar_chart(probs_df.set_index('Class'))
"""

with open(f"{DEPLOY_DIR}/app.py", "w") as f:
    f.write(app_code)
print("✅ Generated app.py")

✅ Generated app.py


In [6]:
# -----------------------------------------------------------
# 6. REQUIREMENTS FILE
# -----------------------------------------------------------
requirements = """
streamlit
torch
torchvision
numpy
pillow
pandas
"""
with open(f"{DEPLOY_DIR}/requirements.txt", "w") as f:
    f.write(requirements)
print("✅ Generated requirements.txt")

✅ Generated requirements.txt
